In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [15]:
# Environment & Directory Setup

import os, sys, json, csv, zipfile, shutil
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Set main directory name to results_food101_v3 to match results_food11_v3
OUT_DIR = '/kaggle/working/results_food101_v3'
ERR_DIR = f'{OUT_DIR}/error_analysis'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(ERR_DIR, exist_ok=True)

print(" Output directory set to:", OUT_DIR)

 Output directory set to: /kaggle/working/results_food101_v3


In [16]:
# Locate Output Files & Load Histories

# Explicitly search root input directory recursively
SEARCH_BASE = '/kaggle/input/datasets/muhammadahmadulster/food-101-model-outputs-v3'

def find_file(filename):
    # Fallback to general /kaggle/input or /kaggle/working if custom base isn't active
    bases = [SEARCH_BASE, '/kaggle/input', '/kaggle/working']
    for base in bases:
        if os.path.exists(base):
            for root, _, files in os.walk(base):
                if filename in files:
                    found_path = os.path.join(root, filename)
                    return found_path
    raise FileNotFoundError(f"Could not locate '{filename}' in {bases}.")

model_keys = [
    ('EfficientNet-B0', 'effnet_history.json', 'EfficientNet-B0_test_predictions.csv'),
    ('ViT-B/16',        'vit_history.json',    'ViT-B_16_test_predictions.csv'),
    ('CoAtNet-0',       'coatnet_history.json', 'CoAtNet-0_test_predictions.csv')
]

histories = {}
for name, hist_file, pred_file in model_keys:
    hp = find_file(hist_file)
    pp = find_file(pred_file)
    with open(hp, 'r') as f:
        histories[name] = json.load(f)
    print(f"✓ Found {name}:")
    print(f"   - History: {hp}")
    print(f"   - Predictions: {pp}")

print("\n All 3 model histories and prediction logs located successfully!\n")

✓ Found EfficientNet-B0:
   - History: /kaggle/input/datasets/muhammadahmadulster/food-101-model-outputs-v3/food101_effnet_results/results_effnet/effnet_history.json
   - Predictions: /kaggle/input/datasets/muhammadahmadulster/food-101-model-outputs-v3/food101_effnet_results/results_effnet/EfficientNet-B0_test_predictions.csv
✓ Found ViT-B/16:
   - History: /kaggle/input/datasets/muhammadahmadulster/food-101-model-outputs-v3/food101_vit_results/results_vit/vit_history.json
   - Predictions: /kaggle/input/datasets/muhammadahmadulster/food-101-model-outputs-v3/food101_vit_results/results_vit/ViT-B_16_test_predictions.csv
✓ Found CoAtNet-0:
   - History: /kaggle/input/datasets/muhammadahmadulster/food-101-model-outputs-v3/food101_coatnet_results/results_coatnet/coatnet_history.json
   - Predictions: /kaggle/input/datasets/muhammadahmadulster/food-101-model-outputs-v3/food101_coatnet_results/results_coatnet/CoAtNet-0_test_predictions.csv

 All 3 model histories and prediction logs located 

In [17]:
# Load Prediction Logs & Compute Metrics

results = []
all_confused = {}
predictions_df = {}

for name, _, pred_file in model_keys:
    pred_path = find_file(pred_file)
    df = pd.read_csv(pred_path)
    predictions_df[name] = df
    
    y_true, y_pred = df['true_label'].values, df['pred_label'].values
    acc = accuracy_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100
    kappa = cohen_kappa_score(y_true, y_pred)
    
    results.append({
        'condition': f'{name} Food-101',
        'top1': acc,
        'f1': f1,
        'kappa': kappa,
        'pred_log_path': pred_path
    })
    
    # Identify misclassified samples for error analysis
    mis = df[df['true_label'] != df['pred_label']]
    pair_counts = mis.groupby(['true_class_name', 'pred_class_name']).size().reset_index(name='count').sort_values('count', ascending=False)
    safe_name = name.replace(' ', '_').replace('/', '_')
    all_confused[name] = (pair_counts, mis, safe_name)
    
    print(f"[{name}] Top-1 Acc: {acc:.2f}% | Macro F1: {f1:.2f}% | Kappa: {kappa:.4f}")

print("\n Prediction logs processed and metrics compiled successfully!\n")

[EfficientNet-B0] Top-1 Acc: 82.50% | Macro F1: 82.48% | Kappa: 0.8233
[ViT-B/16] Top-1 Acc: 84.39% | Macro F1: 84.39% | Kappa: 0.8423
[CoAtNet-0] Top-1 Acc: 86.01% | Macro F1: 86.02% | Kappa: 0.8587

 Prediction logs processed and metrics compiled successfully!



In [18]:
# Plot Training Curves Comparison

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Food-101 (v3) — Training Curves Comparison\nPhase 1: 10ep | Phase 2: EarlyStopping(patience=5)',
             fontsize=12, fontweight='bold')

colours = ['#c0392b', '#2980b9', '#8e44ad']

for (name, h), col in zip(histories.items(), colours):
    val_loss = h.get('val_loss', [])
    val_acc = h.get('val_fine', h.get('val_acc', h.get('val_accuracy', [])))
    
    axes[0].plot(val_loss, label=name, color=col, linewidth=2)
    axes[1].plot(val_acc, label=name, color=col, linewidth=2)

for ax, title, ylabel in [(axes[0], 'Validation Loss', 'Loss'), (axes[1], 'Val Fine Top-1 (%)', 'Accuracy (%)')]:
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
curve_plot_path = f'{OUT_DIR}/food101_v3_training_curves.png'
plt.savefig(curve_plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f" Training curves saved to: {curve_plot_path}\n")

 Training curves saved to: /kaggle/working/results_food101_v3/food101_v3_training_curves.png



In [19]:
# Plot Model Comparison Bar Chart

fig, ax = plt.subplots(figsize=(8, 5))
models = [r['condition'].replace(' Food-101', '') for r in results]
accs = [r['top1'] for r in results]

bars = ax.bar(models, accs, color=colours, edgecolor='white', width=0.45)
ax.set_title('Food-101 Test Accuracy — 3 Models Comparison', fontsize=12, fontweight='bold')
ax.set_ylabel('Top-1 Accuracy (%)')
ax.set_ylim(0, 100)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, v in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2, f'{v:.2f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
bar_plot_path = f'{OUT_DIR}/food101_v3_comparison.png'
plt.savefig(bar_plot_path, dpi=300, bbox_inches='tight')
plt.close()

print(f" Comparison bar chart saved to: {bar_plot_path}\n")

 Comparison bar chart saved to: /kaggle/working/results_food101_v3/food101_v3_comparison.png



In [20]:
#  Plot Confusion Matrices

for name, df in predictions_df.items():
    safe_name = name.replace(' ', '_').replace('/', '_')
    y_true, y_pred = df['true_label'].values, df['pred_label'].values
    
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, cmap='Blues', cbar=True, xticklabels=False, yticklabels=False, ax=ax)
    ax.set_title(f'{name} — Food-101 Fine Confusion Matrix (101 Classes)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    
    cm_path = f'{OUT_DIR}/{safe_name}_Food-101_fine_cm.png'
    plt.tight_layout()
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved fine confusion matrix for {name} -> {cm_path}")

print("Confusion matrix plots generated for all models!\n")

Saved fine confusion matrix for EfficientNet-B0 -> /kaggle/working/results_food101_v3/EfficientNet-B0_Food-101_fine_cm.png
Saved fine confusion matrix for ViT-B/16 -> /kaggle/working/results_food101_v3/ViT-B_16_Food-101_fine_cm.png
Saved fine confusion matrix for CoAtNet-0 -> /kaggle/working/results_food101_v3/CoAtNet-0_Food-101_fine_cm.png
Confusion matrix plots generated for all models!



In [21]:
#  Error Analysis & Per-Class Recall Export

summary_rows = []

for name, (pair_counts, mis, safe_name) in all_confused.items():
    df = predictions_df[name]
    
    # 1. Export Confused Pairs CSV
    pair_path = f'{ERR_DIR}/{safe_name}_confused_pairs.csv'
    pair_counts.to_csv(pair_path, index=False)
    
    # 2. Export Copy of Test Predictions CSV inside error_analysis/
    pred_copy_path = f'{ERR_DIR}/{safe_name}_test_predictions.csv'
    df.to_csv(pred_copy_path, index=False)
    
    # 3. Compute and Export Per-Class Recall CSV
    class_stats = []
    for cls_name in sorted(df['true_class_name'].unique()):
        cls_df = df[df['true_class_name'] == cls_name]
        total = len(cls_df)
        correct = len(cls_df[cls_df['true_class_name'] == cls_df['pred_class_name']])
        recall = round(100 * correct / total, 2) if total > 0 else 0.0
        class_stats.append({
            'class_name': cls_name,
            'total_samples': total,
            'correct_samples': correct,
            'recall_pct': recall
        })
    
    recall_df = pd.DataFrame(class_stats)
    recall_csv_path = f'{ERR_DIR}/{safe_name}_per_class_recall.csv'
    recall_df.to_csv(recall_csv_path, index=False)
    
    # 4. Add to Error Summary Table
    summary_rows.append({
        'model': name,
        'total_samples': len(df),
        'misclassified': len(mis),
        'error_rate_pct': round(100 * len(mis) / len(df), 2)
    })
    print(f"[{name}] Exported predictions, confused pairs, and per-class recall.")

# Export Error Analysis Summary CSV
summary_df = pd.DataFrame(summary_rows)
summary_csv_path = f'{ERR_DIR}/error_analysis_summary.csv'
summary_df.to_csv(summary_csv_path, index=False)

print(f"\n All error analysis files exported to {ERR_DIR}\n")

[EfficientNet-B0] Exported predictions, confused pairs, and per-class recall.
[ViT-B/16] Exported predictions, confused pairs, and per-class recall.
[CoAtNet-0] Exported predictions, confused pairs, and per-class recall.

 All error analysis files exported to /kaggle/working/results_food101_v3/error_analysis



In [22]:
# Plot Coarse Confusion Matrices

# Redefine taxonomy mapping for Notebook 4
COARSE_GROUPS = {
    0: 'Desserts & Sweets', 1: 'Meat & Poultry', 2: 'Seafood',
    3: 'Pizza & Pasta', 4: 'Soups & Stews', 5: 'Salads & Vegetables',
    6: 'Fast Food & Sandwiches', 7: 'Breakfast & Eggs', 8: 'Asian & International'
}

COARSE_MAP = {
    'apple_pie': 0, 'baklava': 0, 'beignets': 0, 'bread_pudding': 0, 'cannoli': 0,
    'carrot_cake': 0, 'cheesecake': 0, 'chocolate_cake': 0, 'chocolate_mousse': 0,
    'churros': 0, 'creme_brulee': 0, 'cup_cakes': 0, 'donuts': 0, 'frozen_yogurt': 0,
    'ice_cream': 0, 'macarons': 0, 'panna_cotta': 0, 'red_velvet_cake': 0,
    'strawberry_shortcake': 0, 'tiramisu': 0,
    'baby_back_ribs': 1, 'beef_carpaccio': 1, 'beef_tartare': 1, 'chicken_curry': 1,
    'chicken_wings': 1, 'filet_mignon': 1, 'foie_gras': 1, 'peking_duck': 1,
    'pork_chop': 1, 'prime_rib': 1, 'steak': 1,
    'ceviche': 2, 'crab_cakes': 2, 'fish_and_chips': 2, 'fried_calamari': 2,
    'grilled_salmon': 2, 'lobster_bisque': 2, 'lobster_roll_sandwich': 2,
    'mussels': 2, 'oysters': 2, 'sashimi': 2, 'scallops': 2,
    'shrimp_and_grits': 2, 'tuna_tartare': 2,
    'cheese_pizza': 3, 'gnocchi': 3, 'lasagna': 3, 'macaroni_and_cheese': 3,
    'pizza': 3, 'ravioli': 3, 'risotto': 3, 'spaghetti_bolognese': 3,
    'spaghetti_carbonara': 3,
    'clam_chowder': 4, 'french_onion_soup': 4, 'hot_and_sour_soup': 4,
    'miso_soup': 4, 'pho': 4, 'ramen': 4,
    'beet_salad': 5, 'caesar_salad': 5, 'caprese_salad': 5, 'edamame': 5,
    'greek_salad': 5, 'guacamole': 5, 'hummus': 5, 'seaweed_salad': 5,
    'breakfast_burrito': 6, 'bruschetta': 6, 'club_sandwich': 6,
    'croque_madame': 6, 'french_fries': 6, 'garlic_bread': 6,
    'grilled_cheese_sandwich': 6, 'hamburger': 6, 'hot_dog': 6, 'nachos': 6,
    'onion_rings': 6, 'pulled_pork_sandwich': 6, 'tacos': 6,
    'deviled_eggs': 7, 'eggs_benedict': 7, 'french_toast': 7,
    'huevos_rancheros': 7, 'pancakes': 7, 'waffles': 7,
    'bibimbap': 8, 'dumplings': 8, 'escargots': 8, 'falafel': 8,
    'fried_rice': 8, 'pad_thai': 8, 'paella': 8, 'poutine': 8, 'samosa': 8,
    'spring_rolls': 8, 'sushi': 8, 'takoyaki': 8,
}

for name, df in predictions_df.items():
    safe_name = name.replace(' ', '_').replace('/', '_')
    
    # Map fine names to coarse IDs
    y_true_coarse = df['true_class_name'].map(lambda x: COARSE_MAP.get(x, 8)).values
    y_pred_coarse = df['pred_class_name'].map(lambda x: COARSE_MAP.get(x, 8)).values
    
    cm_coarse = confusion_matrix(y_true_coarse, y_pred_coarse)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm_coarse, annot=True, fmt='d', cmap='Oranges',
                xticklabels=[COARSE_GROUPS[i] for i in range(9)],
                yticklabels=[COARSE_GROUPS[i] for i in range(9)], ax=ax)
    
    ax.set_title(f'{name} — Food-101 Coarse Confusion Matrix (9 Categories)', fontweight='bold')
    ax.set_xlabel('Predicted Category')
    ax.set_ylabel('True Category')
    plt.xticks(rotation=45, ha='right')
    
    coarse_cm_path = f'{OUT_DIR}/{safe_name}_Food-101_coarse_cm.png'
    plt.tight_layout()
    plt.savefig(coarse_cm_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved coarse confusion matrix -> {coarse_cm_path}")

print("Coarse Confusion Matrices generated!\n")

Saved coarse confusion matrix -> /kaggle/working/results_food101_v3/EfficientNet-B0_Food-101_coarse_cm.png
Saved coarse confusion matrix -> /kaggle/working/results_food101_v3/ViT-B_16_Food-101_coarse_cm.png
Saved coarse confusion matrix -> /kaggle/working/results_food101_v3/CoAtNet-0_Food-101_coarse_cm.png
Coarse Confusion Matrices generated!



In [25]:
# Visual Misclassification Contact Sheets (Fixed Path Finder)

from PIL import Image

# Locate the root images directory dynamically in Kaggle
IMG_SEARCH_ROOTS = [
    '/kaggle/input/datasets/srujanesanakarra/food101/food-101/images',
    '/kaggle/input/food101/food-101/images',
    '/kaggle/input/food-101/images',
    '/kaggle/input/food101/images'
]

def find_image_path(sample_id, true_class):
    # Try direct extensions and subfolder structures
    candidates = [
        f"{sample_id}",
        f"{sample_id}.jpg",
        f"{true_class}/{sample_id}.jpg",
        f"{true_class}/{sample_id}",
    ]
    
    for s_root in IMG_SEARCH_ROOTS:
        if os.path.exists(s_root):
            for cand in candidates:
                full_p = os.path.join(s_root, cand)
                if os.path.exists(full_p):
                    return full_p
                    
    # Fallback recursive search if path structured differently
    for s_root in ['/kaggle/input']:
        if os.path.exists(s_root):
            clean_id = str(sample_id).split('/')[-1].replace('.jpg', '')
            for root, _, files in os.walk(s_root):
                for f in files:
                    if clean_id in f and f.endswith(('.jpg', '.jpeg', '.png')):
                        return os.path.join(root, f)
    return None

for name, (pair_counts, mis, safe_name) in all_confused.items():
    if len(pair_counts) == 0:
        continue
        
    top_true = pair_counts.iloc[0]['true_class_name']
    top_pred = pair_counts.iloc[0]['pred_class_name']
    
    pair_df = mis[(mis['true_class_name'] == top_true) & (mis['pred_class_name'] == top_pred)]
    samples = pair_df.head(10)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle(f'{name} - Top Confusion: True "{top_true}" vs Predicted "{top_pred}"', fontsize=14, fontweight='bold')
    axes = axes.flatten()
    
    loaded_count = 0
    for ax, (_, row) in zip(axes, samples.iterrows()):
        img_path = find_image_path(row['sample_id'], row['true_class_name'])
        
        if img_path and os.path.exists(img_path):
            try:
                img = Image.open(img_path)
                ax.imshow(img)
                ax.set_title(f"Conf: {row['confidence']:.2f}", fontsize=10, color='darkred', fontweight='bold')
                loaded_count += 1
            except Exception:
                ax.set_title("Read Error", fontsize=10)
        else:
            ax.set_title("Path Not Found", fontsize=10)
        ax.axis('off')
        
    for i in range(len(samples), 10):
        axes[i].axis('off')
        
    plt.tight_layout()
    contact_sheet_path = f'{ERR_DIR}/{safe_name}_top_confused_pair_contact_sheet.png'
    plt.savefig(contact_sheet_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"[{name}] Found {loaded_count}/{len(samples)} images -> Saved: {contact_sheet_path}")

print("\n Visual Contact Sheets generated!\n")

[EfficientNet-B0] Found 10/10 images -> Saved: /kaggle/working/results_food101_v3/error_analysis/EfficientNet-B0_top_confused_pair_contact_sheet.png
[ViT-B/16] Found 10/10 images -> Saved: /kaggle/working/results_food101_v3/error_analysis/ViT-B_16_top_confused_pair_contact_sheet.png
[CoAtNet-0] Found 10/10 images -> Saved: /kaggle/working/results_food101_v3/error_analysis/CoAtNet-0_top_confused_pair_contact_sheet.png

 Visual Contact Sheets generated!



In [26]:
#  Export Master CSV, Copy Histories & Build Final ZIP

# 1. Save Master Comparative Results CSV
csv_path = f'{OUT_DIR}/FOOD101_V3_RESULTS.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['condition', 'top1', 'f1', 'kappa'])
    writer.writeheader()
    for r in results:
        writer.writerow({
            'condition': r['condition'],
            'top1': round(r['top1'], 2),
            'f1': round(r['f1'], 2),
            'kappa': round(r['kappa'], 4)
        })

# 2. Copy History JSON files and create epoch_counts.json in root OUT_DIR
hist_filename_map = {
    'EfficientNet-B0': 'effnet_history.json',
    'ViT-B/16': 'vit_history.json',
    'CoAtNet-0': 'coatnet_history.json'
}

epoch_counts = {}

for name, hist_dict in histories.items():
    json_filename = hist_filename_map[name]
    dst_path = f'{OUT_DIR}/{json_filename}'
    
    with open(dst_path, 'w') as f:
        json.dump(hist_dict, f, indent=4)
        
    val_loss_list = hist_dict.get('val_loss', [])
    epoch_counts[name] = len(val_loss_list)

# Export epoch_counts.json
epoch_json_path = f'{OUT_DIR}/epoch_counts.json'
with open(epoch_json_path, 'w') as f:
    json.dump(epoch_counts, f, indent=4)

print("✓ Saved histories and epoch_counts.json to root folder.")

# 3. Create Master ZIP Archive with exact internal folder hierarchy
zip_path = '/kaggle/working/results_food101_v3.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        for file in files:
            fp = os.path.join(root, file)
            # Retain 'results_food101_v3/...' top folder inside zip
            arcname = os.path.relpath(fp, '/kaggle/working')
            zf.write(fp, arcname)

print(f"\n Master ZIP ready at: {zip_path}\n")

# Verify all files inside the generated ZIP archive
with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = sorted(zf.namelist())
    print(f"=== ZIP Contents ({len(zip_contents)} files total) ===")
    for c in zip_contents:
        print(" -", c)

from IPython.display import FileLink, display
display(FileLink('results_food101_v3.zip'))

✓ Saved histories and epoch_counts.json to root folder.

 Master ZIP ready at: /kaggle/working/results_food101_v3.zip

=== ZIP Contents (26 files total) ===
 - results_food101_v3/CoAtNet-0_Food-101_coarse_cm.png
 - results_food101_v3/CoAtNet-0_Food-101_fine_cm.png
 - results_food101_v3/EfficientNet-B0_Food-101_coarse_cm.png
 - results_food101_v3/EfficientNet-B0_Food-101_fine_cm.png
 - results_food101_v3/FOOD101_V3_RESULTS.csv
 - results_food101_v3/ViT-B_16_Food-101_coarse_cm.png
 - results_food101_v3/ViT-B_16_Food-101_fine_cm.png
 - results_food101_v3/coatnet_history.json
 - results_food101_v3/effnet_history.json
 - results_food101_v3/epoch_counts.json
 - results_food101_v3/error_analysis/CoAtNet-0_confused_pairs.csv
 - results_food101_v3/error_analysis/CoAtNet-0_per_class_recall.csv
 - results_food101_v3/error_analysis/CoAtNet-0_test_predictions.csv
 - results_food101_v3/error_analysis/CoAtNet-0_top_confused_pair_contact_sheet.png
 - results_food101_v3/error_analysis/EfficientNet-B0_c

/kaggle/working/results_food101_v3.zip